# Abhishek Das (G25AIT1005)

# Sanskrit → English Neural Machine Translation
**NLU Assignment 2** — Custom Transformer sequence-to-sequence model (implemented from scratch in PyTorch)

**Pipeline:** data cleaning → SentencePiece BPE tokenization → Transformer encoder-decoder (pre-norm, tied output embeddings) → label-smoothed training with warmup/inverse-sqrt LR → beam-search decoding → BLEU / BERTScore / efficiency evaluation → `submission.csv`.

No pre-trained models are used for translation itself; BERTScore (evaluation only) internally uses a pre-trained RoBERTa model, as per the official `bert-score` library required by the assignment.

## 0. Install dependencies

In [2]:
# Installation steps for all required dependencies
!pip install -q torch sentencepiece nltk bert-score pandas matplotlib numpy


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports, configuration & reproducibility
All hyperparameters live in one `CFG` class. Seeds are fixed for full reproducibility.

In [3]:
import os, math, time, json, random, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

# ------------------------------------------------------------
# 0. Config & reproducibility
# ------------------------------------------------------------
TINY = False  # set True for a quick smoke-test run

class CFG:
    seed          = 42
    data_dir      = "."
    work_dir      = "work"
    # tokenization
    src_vocab     = 6000 if not TINY else 400
    tgt_vocab     = 6000 if not TINY else 400
    max_len       = 128
    # model
    d_model       = 256 if not TINY else 64
    n_heads       = 4
    enc_layers    = 3 if not TINY else 1
    dec_layers    = 3 if not TINY else 1
    d_ff          = 1024 if not TINY else 128
    dropout       = 0.15
    # training
    batch_size    = 64 if not TINY else 8
    epochs        = 70 if not TINY else 2
    lr_peak       = 5e-4
    warmup_steps  = 1000 if not TINY else 10
    label_smooth  = 0.1
    grad_clip     = 1.0
    # decoding
    beam_size     = 4
    len_penalty   = 0.6
    gen_max_len   = 120 if not TINY else 30
    # eval cadence
    eval_every    = 2          # epochs between dev-BLEU checks
    dev_bleu_n    = 300        # dev subset size for early-model selection

cfg = CFG()
os.makedirs(cfg.work_dir, exist_ok=True)

random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
print("Device:", device)


Device: cuda


## 2. Data loading & cleaning
The Sanskrit and English CSVs are aligned on `Source_id`. Cleaning: NFC unicode normalization, zero-width-joiner removal, whitespace collapsing.

**Set `CFG.data_dir` to the folder containing the six CSV files.**

In [4]:
def load_pairs(split, n):
    sa = pd.read_csv(f"{cfg.data_dir}/{split}_sa_{n}.csv", encoding="utf-8-sig")
    en = pd.read_csv(f"{cfg.data_dir}/{split}_en_{n}.csv", encoding="utf-8-sig")
    df = sa.merge(en, on="Source_id", how="inner")
    df["Sentence_sa"] = df["Sentence_sa"].astype(str).map(clean_text)
    df["Sentence_en"] = df["Sentence_en"].astype(str).map(clean_text)
    return df

def clean_text(s):
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u200d", "").replace("\u200c", "")   # zero-width joiners
    return " ".join(s.split()).strip()

train_df = load_pairs("train", 10000)
dev_df   = load_pairs("dev",   1000)
test_df  = load_pairs("test",  1000)
if TINY:
    train_df = train_df.head(200); dev_df = dev_df.head(30); test_df = test_df.head(30)
print(len(train_df), "train |", len(dev_df), "dev |", len(test_df), "test")


10000 train | 1000 dev | 1000 test


## 3. Subword tokenization — SentencePiece BPE
Sanskrit is morphologically rich (heavy sandhi/compounding), so word-level vocabularies explode and suffer from out-of-vocabulary tokens. BPE subwords solve both problems. Separate 6k vocabularies are trained for Sanskrit (Devanagari) and English, **on the training split only** (no leakage).

In [5]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3

def train_spm(sentences, prefix, vocab_size):
    txt = f"{cfg.work_dir}/{prefix}.txt"
    with open(txt, "w", encoding="utf-8") as f:
        f.write("\n".join(sentences))
    spm.SentencePieceTrainer.train(
        input=txt, model_prefix=f"{cfg.work_dir}/{prefix}",
        vocab_size=vocab_size, model_type="bpe",
        character_coverage=0.9995,
        pad_id=PAD, bos_id=BOS, eos_id=EOS, unk_id=UNK,
    )
    sp = spm.SentencePieceProcessor(model_file=f"{cfg.work_dir}/{prefix}.model")
    return sp

sp_sa = train_spm(train_df.Sentence_sa.tolist(), "spm_sa", cfg.src_vocab)
sp_en = train_spm(train_df.Sentence_en.tolist(), "spm_en", cfg.tgt_vocab)
print("src vocab:", sp_sa.get_piece_size(), "| tgt vocab:", sp_en.get_piece_size())

def encode_src(s): return [BOS] + sp_sa.encode(s)[: cfg.max_len - 2] + [EOS]
def encode_tgt(s): return [BOS] + sp_en.encode(s)[: cfg.max_len - 2] + [EOS]


src vocab: 6000 | tgt vocab: 6000


## 4. Dataset & DataLoader

In [6]:
class NMTDataset(Dataset):
    def __init__(self, df):
        self.src = [encode_src(s) for s in df.Sentence_sa]
        self.tgt = [encode_tgt(s) for s in df.Sentence_en]
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        return torch.tensor(self.src[i]), torch.tensor(self.tgt[i])

def collate(batch):
    srcs, tgts = zip(*batch)
    src = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD)
    tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD)
    return src, tgt

train_loader = DataLoader(NMTDataset(train_df), batch_size=cfg.batch_size,
                          shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(NMTDataset(dev_df), batch_size=cfg.batch_size,
                          shuffle=False, collate_fn=collate)


## 5. Model — Transformer encoder-decoder from scratch
Custom implementation (no `nn.Transformer`): sinusoidal positional encoding (parameter-free), multi-head scaled-dot-product attention, GELU feed-forward blocks, **pre-norm** residual connections (more stable for small datasets), and **weight tying** between the target embedding and the output projection (saves ~1.5M parameters — helps the efficiency score).

Initialization: Xavier-uniform for matrices; embeddings ~ N(0, d^-0.5) so that after the √d scaling the input to the network has unit variance (critical for stable training with tied embeddings).

In [7]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (parameter-free)."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):                       # x: (B, T, D)
        return x + self.pe[:, : x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B, Tq, D = q.shape
        Tk = k.size(1)
        q = self.wq(q).view(B, Tq, self.h, self.dk).transpose(1, 2)   # (B,h,Tq,dk)
        k = self.wk(k).view(B, Tk, self.h, self.dk).transpose(1, 2)
        v = self.wv(v).view(B, Tk, self.h, self.dk).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)          # (B,h,Tq,Tk)
        if mask is not None:                     # mask==True -> blocked
            att = att.masked_fill(mask, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, D)
        return self.wo(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, src_mask):              # pre-norm residual blocks
        h = self.n1(x); x = x + self.drop(self.attn(h, h, h, src_mask))
        h = self.n2(x); x = x + self.drop(self.ff(h))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d_model) for _ in range(3))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mem, tgt_mask, mem_mask):
        h = self.n1(x); x = x + self.drop(self.self_attn(h, h, h, tgt_mask))
        h = self.n2(x); x = x + self.drop(self.cross_attn(h, mem, mem, mem_mask))
        h = self.n3(x); x = x + self.drop(self.ff(h))
        return x

class Seq2SeqTransformer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d_model = c.d_model
        self.src_emb = nn.Embedding(c.src_vocab, c.d_model, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(c.tgt_vocab, c.d_model, padding_idx=PAD)
        self.pos = PositionalEncoding(c.d_model, max_len=max(c.max_len, c.gen_max_len) + 8)
        self.enc = nn.ModuleList([EncoderLayer(c.d_model, c.n_heads, c.d_ff, c.dropout)
                                  for _ in range(c.enc_layers)])
        self.dec = nn.ModuleList([DecoderLayer(c.d_model, c.n_heads, c.d_ff, c.dropout)
                                  for _ in range(c.dec_layers)])
        self.enc_norm = nn.LayerNorm(c.d_model)
        self.dec_norm = nn.LayerNorm(c.d_model)
        self.out = nn.Linear(c.d_model, c.tgt_vocab, bias=False)
        self.out.weight = self.tgt_emb.weight        # weight tying (fewer params)
        self.drop = nn.Dropout(c.dropout)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        # embeddings ~ N(0, d^-0.5) so that emb * sqrt(d) has unit variance
        nn.init.normal_(self.src_emb.weight, mean=0.0, std=self.d_model ** -0.5)
        nn.init.normal_(self.tgt_emb.weight, mean=0.0, std=self.d_model ** -0.5)
        with torch.no_grad():
            self.src_emb.weight[PAD].zero_()
            self.tgt_emb.weight[PAD].zero_()

    def make_pad_mask(self, seq):                    # (B,1,1,T) True where PAD
        return (seq == PAD).unsqueeze(1).unsqueeze(2)

    def make_causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

    def encode(self, src):
        src_mask = self.make_pad_mask(src)
        x = self.drop(self.pos(self.src_emb(src) * math.sqrt(self.d_model)))
        for layer in self.enc:
            x = layer(x, src_mask)
        return self.enc_norm(x), src_mask

    def decode(self, tgt, mem, mem_mask):
        T = tgt.size(1)
        tgt_mask = self.make_causal_mask(T, tgt.device) | self.make_pad_mask(tgt)
        x = self.drop(self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model)))
        for layer in self.dec:
            x = layer(x, mem, tgt_mask, mem_mask)
        return self.out(self.dec_norm(x))

    def forward(self, src, tgt_in):
        mem, mem_mask = self.encode(src)
        return self.decode(tgt_in, mem, mem_mask)

model = Seq2SeqTransformer(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {n_params:,}")


Total trainable parameters: 8,602,624


## 6. Training setup + decoding (greedy & beam search)
- **Loss:** cross-entropy with label smoothing 0.1 (ignoring PAD)
- **Optimizer:** AdamW, β=(0.9, 0.98), weight decay 1e-4
- **Schedule:** linear warmup (1000 steps) → inverse-sqrt decay
- **Beam search:** batched, beam=4, GNMT length penalty ((5+len)/6)^0.6
- **BLEU:** default NLTK `corpus_bleu` with no weights argument, as required

In [8]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=cfg.label_smooth)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr_peak, betas=(0.9, 0.98),
                              eps=1e-9, weight_decay=1e-4)

def lr_lambda(step):                              # linear warmup, inverse-sqrt decay
    step = max(step, 1)
    return min(step / cfg.warmup_steps, math.sqrt(cfg.warmup_steps / step))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def greedy_decode(src_batch, max_len=None):
    """Batched greedy decoding."""
    model.eval()
    max_len = max_len or cfg.gen_max_len
    src = src_batch.to(device)
    mem, mem_mask = model.encode(src)
    B = src.size(0)
    ys = torch.full((B, 1), BOS, dtype=torch.long, device=device)
    done = torch.zeros(B, dtype=torch.bool, device=device)
    for _ in range(max_len - 1):
        logits = model.decode(ys, mem, mem_mask)[:, -1]
        nxt = logits.argmax(-1)
        nxt = nxt.masked_fill(done, PAD)
        ys = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
        done |= nxt == EOS
        if done.all(): break
    return ys

@torch.no_grad()
def beam_decode(src_batch, beam=None, alpha=None, max_len=None):
    """Batched beam search with length penalty ((5+len)/6)^alpha."""
    model.eval()
    beam  = beam or cfg.beam_size
    alpha = cfg.len_penalty if alpha is None else alpha
    max_len = max_len or cfg.gen_max_len
    src = src_batch.to(device)
    B = src.size(0)
    mem, mem_mask = model.encode(src)
    D, S = mem.size(-1), mem.size(1)
    # expand memory to (B*beam, S, D)
    mem      = mem.unsqueeze(1).expand(B, beam, S, D).reshape(B * beam, S, D)
    mem_mask = mem_mask.expand(B, 1, 1, S).unsqueeze(1) \
                       .expand(B, beam, 1, 1, S).reshape(B * beam, 1, 1, S)
    ys     = torch.full((B * beam, 1), BOS, dtype=torch.long, device=device)
    scores = torch.full((B, beam), float("-inf"), device=device)
    scores[:, 0] = 0.0                            # only first beam live at step 0
    finished = torch.zeros(B * beam, dtype=torch.bool, device=device)
    for step in range(max_len - 1):
        logits = model.decode(ys, mem, mem_mask)[:, -1]          # (B*beam, V)
        logp = F.log_softmax(logits, dim=-1)
        V = logp.size(-1)
        # finished hypotheses only extend with PAD at zero cost
        logp[finished] = float("-inf")
        logp[finished, PAD] = 0.0
        cand = scores.view(-1, 1) + logp                          # (B*beam, V)
        cand = cand.view(B, beam * V)
        top_sc, top_ix = cand.topk(beam, dim=-1)                  # (B, beam)
        beam_ix, tok_ix = top_ix // V, top_ix % V
        flat = (torch.arange(B, device=device).unsqueeze(1) * beam + beam_ix).view(-1)
        ys = torch.cat([ys[flat], tok_ix.view(-1, 1)], dim=1)
        finished = finished[flat] | (tok_ix.view(-1) == EOS)
        scores = top_sc
        if finished.all(): break
    # length-penalised selection of best beam per source
    lengths = (ys != PAD).sum(1).float().view(B, beam)
    lp = ((5.0 + lengths) / 6.0) ** alpha
    best = (scores / lp).argmax(dim=-1)                           # (B,)
    idx = torch.arange(B, device=device) * beam + best
    return ys.view(B * beam, -1)[idx]

def ids_to_text(ys):
    outs = []
    for row in ys.tolist():
        toks = []
        for t in row[1:]:
            if t == EOS: break
            if t not in (PAD, BOS): toks.append(t)
        outs.append(sp_en.decode(toks))
    return outs

def translate(sentences, decode_fn=greedy_decode, batch_size=64):
    preds = []
    for i in range(0, len(sentences), batch_size):
        chunk = sentences[i : i + batch_size]
        ids = [torch.tensor(encode_src(s)) for s in chunk]
        src = nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=PAD)
        preds += ids_to_text(decode_fn(src))
    return preds

# ---- BLEU helpers (default NLTK BLEU, no weights argument) ----
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import corpus_bleu
from nltk.tokenize import word_tokenize

def bleu_score(refs, hyps):
    return corpus_bleu([[word_tokenize(r)] for r in refs],
                       [word_tokenize(h) for h in hyps])


## 7. Training loop
Dev BLEU (greedy, 300-sentence subset) is computed every 2 epochs for model selection; the best checkpoint is kept. On an RTX 4070 this takes roughly 5–10 minutes.

In [20]:
history = {"train_loss": [], "dev_loss": [], "dev_bleu": []}
best_bleu, best_path = -1.0, f"{cfg.work_dir}/best_model.pt"

for epoch in range(1, cfg.epochs + 1):
    model.train()
    tot, ntok = 0.0, 0
    t0 = time.time()
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        logits = model(src, tgt[:, :-1])
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step(); scheduler.step()
        k = (tgt[:, 1:] != PAD).sum().item()
        tot += loss.item() * k; ntok += k
    train_loss = tot / ntok

    model.eval(); dtot, dtok = 0.0, 0
    with torch.no_grad():
        for src, tgt in dev_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1])
            l = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            k = (tgt[:, 1:] != PAD).sum().item()
            dtot += l.item() * k; dtok += k
    dev_loss = dtot / dtok
    history["train_loss"].append(train_loss); history["dev_loss"].append(dev_loss)

    msg = (f"epoch {epoch:02d} | train {train_loss:.3f} | dev {dev_loss:.3f} "
           f"| lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
    if epoch % cfg.eval_every == 0 or epoch == cfg.epochs:
        sub = dev_df.head(cfg.dev_bleu_n)
        db = bleu_score(sub.Sentence_en.tolist(), translate(sub.Sentence_sa.tolist()))
        history["dev_bleu"].append((epoch, db))
        msg += f" | dev-BLEU {db:.4f}"
        if db > best_bleu:
            best_bleu = db
            torch.save(model.state_dict(), best_path)
            msg += "  <- saved"
    print(msg)

model.load_state_dict(torch.load(best_path, map_location=device))
print(f"Best dev-BLEU (subset): {best_bleu:.4f}")


epoch 01 | train 8.063 | dev 7.128 | lr 7.85e-05 | 21s


c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


epoch 02 | train 6.825 | dev 6.574 | lr 1.57e-04 | 20s | dev-BLEU 0.0000  <- saved
epoch 03 | train 6.424 | dev 6.221 | lr 2.36e-04 | 19s
epoch 04 | train 6.070 | dev 5.900 | lr 3.14e-04 | 19s | dev-BLEU 0.0191  <- saved
epoch 05 | train 5.756 | dev 5.654 | lr 3.92e-04 | 20s
epoch 06 | train 5.485 | dev 5.470 | lr 4.71e-04 | 20s | dev-BLEU 0.0307  <- saved
epoch 07 | train 5.242 | dev 5.321 | lr 4.77e-04 | 19s
epoch 08 | train 5.008 | dev 5.220 | lr 4.46e-04 | 19s | dev-BLEU 0.0383  <- saved
epoch 09 | train 4.806 | dev 5.137 | lr 4.21e-04 | 20s
epoch 10 | train 4.628 | dev 5.080 | lr 3.99e-04 | 21s | dev-BLEU 0.0452  <- saved
epoch 11 | train 4.466 | dev 5.032 | lr 3.80e-04 | 20s
epoch 12 | train 4.319 | dev 4.993 | lr 3.64e-04 | 19s | dev-BLEU 0.0569  <- saved
epoch 13 | train 4.181 | dev 4.976 | lr 3.50e-04 | 19s
epoch 14 | train 4.060 | dev 4.962 | lr 3.37e-04 | 20s | dev-BLEU 0.0652  <- saved
epoch 15 | train 3.942 | dev 4.946 | lr 3.26e-04 | 18s
epoch 16 | train 3.841 | dev 4.949

## 8. Test-set inference (timed) & `submission.csv`
Efficiency metric 1: total wall-clock time to translate the full test set with beam search. Efficiency metric 2: total trainable parameter count (printed again here).

In [9]:
model.eval()
if device.type == "cuda": torch.cuda.synchronize()
t_start = time.time()
test_preds = translate(test_df.Sentence_sa.tolist(),
                       decode_fn=lambda s: beam_decode(s))
if device.type == "cuda": torch.cuda.synchronize()
inference_time = time.time() - t_start
print(f"Inference time (test set, beam={cfg.beam_size}): {inference_time:.2f} s")
print(f"Total trainable parameters: {n_params:,}")

sub = pd.DataFrame({"Source_id": test_df.Source_id, "Sentence_en": test_preds})
sub.to_csv("submission.csv", index=False, encoding="utf-8")
print("submission.csv written:", sub.shape)


Inference time (test set, beam=4): 102.26 s
Total trainable parameters: 8,602,624
submission.csv written: (1000, 2)


## 9. Evaluation — BLEU & BERTScore (dev + test)
- BLEU: default NLTK `corpus_bleu`, no weights.
- BERTScore: official `bert-score` library, F1 only, `rescale_with_baseline=True`.

In [10]:
dev_preds = translate(dev_df.Sentence_sa.tolist(), decode_fn=lambda s: beam_decode(s))
dev_bleu  = bleu_score(dev_df.Sentence_en.tolist(),  dev_preds)
test_bleu = bleu_score(test_df.Sentence_en.tolist(), test_preds)
print(f"BLEU  dev: {dev_bleu:.4f} | test: {test_bleu:.4f}")

RUN_BERTSCORE = True
if RUN_BERTSCORE:
    from bert_score import score as bert_score_fn
    _, _, f1_dev  = bert_score_fn(dev_preds,  dev_df.Sentence_en.tolist(),
                                  lang="en", rescale_with_baseline=True)
    _, _, f1_test = bert_score_fn(test_preds, test_df.Sentence_en.tolist(),
                                  lang="en", rescale_with_baseline=True)
    bs_dev, bs_test = f1_dev.mean().item(), f1_test.mean().item()
    print(f"BERTScore-F1  dev: {bs_dev:.4f} | test: {bs_test:.4f}")
else:
    bs_dev = bs_test = None

results = {"params": n_params, "inference_time_s": round(inference_time, 2),
           "bleu_dev": round(dev_bleu, 4), "bleu_test": round(test_bleu, 4),
           "bertscore_f1_dev": bs_dev, "bertscore_f1_test": bs_test,
           "beam_size": cfg.beam_size}
with open("results.json", "w") as f: json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))


c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-

BLEU  dev: 0.0000 | test: 0.0000


c:\Users\Abhishek Das\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5677.86it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Con

BERTScore-F1  dev: -0.6295 | test: -0.6256
{
  "params": 8602624,
  "inference_time_s": 102.26,
  "bleu_dev": 0.0,
  "bleu_test": 0.0,
  "bertscore_f1_dev": -0.629494845867157,
  "bertscore_f1_test": -0.6256448030471802,
  "beam_size": 4
}


## 10. Training curves & translation examples (for the report)

In [11]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(history["train_loss"], label="train")
    ax[0].plot(history["dev_loss"], label="dev")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy loss")
    ax[0].set_title("Loss curves"); ax[0].legend(); ax[0].grid(alpha=.3)
    if history["dev_bleu"]:
        xs, ys_ = zip(*history["dev_bleu"])
        ax[1].plot(xs, ys_, marker="o")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("dev BLEU (subset)")
    ax[1].set_title("Dev BLEU"); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig("training_curves.png", dpi=150)
    print("training_curves.png saved")
except Exception as e:
    print("plotting skipped:", e)

print("\n===== Translation examples (test set) =====")
ex_rows = []
for i in random.Random(0).sample(range(len(test_df)), min(8, len(test_df))):
    row = {"source": test_df.Sentence_sa.iloc[i],
           "reference": test_df.Sentence_en.iloc[i],
           "prediction": test_preds[i]}
    ex_rows.append(row)
    print(f"\nSA : {row['source']}\nREF: {row['reference']}\nHYP: {row['prediction']}")
pd.DataFrame(ex_rows).to_csv("examples.csv", index=False, encoding="utf-8")


plotting skipped: name 'history' is not defined

===== Translation examples (test set) =====

SA : कृपया एतौ द्वौ अपि आदेशौ उपयुज्य तस्य प्रयोजनं स्वयं पश्यन्तु।
REF: Please try these two commands and find the output for yourself as an assignment.
HYP: e e e e e e e e e e e e Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Configure Con

## 11. Evaluation on private test set 

In [12]:
# ==== PRIVATE EVALUATION ====
PRIVATE_SA = "test_demo_sa - Sheet1.csv"                 # <- apni Sanskrit file ka EXACT naam
PRIVATE_EN = "test_demo_en - Sheet1.csv"        # <- gold English file

import time, json, pandas as pd, torch

# trained checkpoint load (no retraining)
model.load_state_dict(torch.load(f"{cfg.work_dir}/best_model.pt", map_location=device))
model.eval()

sa_df = pd.read_csv(PRIVATE_SA, encoding="utf-8-sig")
en_df = pd.read_csv(PRIVATE_EN, encoding="utf-8-sig")
df = sa_df.merge(en_df, on="Source_id")
df["Sentence_sa"] = df["Sentence_sa"].astype(str).map(clean_text)
df["Sentence_en"] = df["Sentence_en"].astype(str).map(clean_text)
print(len(df), "sentence pairs")

if device.type == "cuda": torch.cuda.synchronize()
t0 = time.time()
preds = translate(df.Sentence_sa.tolist(), decode_fn=lambda s: beam_decode(s))
if device.type == "cuda": torch.cuda.synchronize()
t_inf = time.time() - t0

bleu = bleu_score(df.Sentence_en.tolist(), preds)
from bert_score import score as bert_score_fn
_, _, f1 = bert_score_fn(preds, df.Sentence_en.tolist(), lang="en", rescale_with_baseline=True)
bs = f1.mean().item()

print("="*50)
print("FORM VALUES")
print(f"BLEU                 : {bleu:.4f}")
print(f"BERT (F1)            : {bs:.4f}")
print(f"Time efficiency (s)  : {t_inf:.2f}")
print(f"Parameter efficiency : {n_params:,}")
print("="*50)

pd.DataFrame({"Source_id": df.Source_id, "Sentence_en": preds}) \
    .to_csv("submission_private.csv", index=False, encoding="utf-8")
json.dump({"bleu": round(bleu,4), "bertscore_f1": round(bs,4),
           "inference_time_s": round(t_inf,2), "params": n_params},
          open("results_private.json","w"), indent=2)
print("saved: submission_private.csv + results_private.json")

100 sentence pairs


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10805.84it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FORM VALUES
BLEU                 : 0.1081
BERT (F1)            : 0.3363
Time efficiency (s)  : 4.95
Parameter efficiency : 8,602,624
saved: submission_private.csv + results_private.json
